In [ ]:
## Initialzing and loading required libraries and subfunctions
import numpy as np
import matplotlib.pyplot as plt
import copy
import yasa
from mne.filter import resample
import pynapple as nap
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import normalize
import requests
from io import BytesIO
import sails
from ssqueezepy import cwt
import re
from scipy.stats import entropy
import os, sys

import scipy
from scipy import signal
from scipy.interpolate import griddata
from scipy.signal import correlate
from scipy.stats import pearsonr
from scipy.fft import fft
from scipy.spatial.distance import euclidean
from scipy.signal import spectrogram
from scipy.io import loadmat
import scipy.fft
import scipy.stats
import scipy.io as sio
from scipy.signal import hilbert

import emd as emd
import emd.sift as sift
import emd.spectra as spectra

from neurodsp.sim import sim_combined
from neurodsp.plts import plot_time_series, plot_timefrequency
from neurodsp.utils import create_times
from neurodsp.timefrequency.wavelets import compute_wavelet_transform
from neurodsp.filt import filter_signal

# Load required libraries
import numpy as np
from scipy.io import loadmat
from scipy.signal import hilbert
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import seaborn as sns
from neurodsp.filt import filter_signal, filter_signal_fir, design_fir_filter
import emd
import pandas as pd
from sklearn.preprocessing import Normalizer
from tqdm import tqdm
import plotly.express as px
import copy
import umap.umap_ as umap
from scipy.spatial import cKDTree
import pickle
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable

## UTILS

for rel in ('src', '../src'):
    p = os.path.abspath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from utils import *
from detect_pt import *

from scipy.io import loadmat
import numpy as np
from neurodsp.filt import filter_signal
import copy
import emd
from scipy.spatial import cKDTree
from tqdm import tqdm

sns.set(style='white', context='notebook')

In [ ]:
path_to_config = '/Users/amir/Desktop/for Abdel/emd_masksift_CA1_config.yml'
config = emd.sift.SiftConfig.from_yaml_file(path_to_config)

# preprocessing

In [ ]:
def extract_lfp_by_pt_intervals(lfp, fs, interval):
    rem_lfp = []
    for ii in range(len(interval)):
        start_idx = int(interval.loc[ii, 'start'] * fs)
        end_idx = int(interval.loc[ii, 'end'] * fs)
        rem_lfp.append(np.array(lfp[start_idx:end_idx]))
    return rem_lfp


In [ ]:
# preprocessing
def extract_cycle_info(imfs, imf_frequencies):

  waveforms = pd.DataFrame()
  all_trials = pd.DataFrame()
  raw_wavelets = []
  all_FPPs = []

  theta_range = [5, 12]
  frequencies = np.arange(15, 141, 1)  # Logarithmically spaced frequencies from 15 to 300 Hz
  angles=np.linspace(-180,180,19)
  fs = 1000

  for idx, imf in enumerate(imfs):
    cycle_data = get_cycle_data(imf[:, 5], fs)

    amp_thresh = np.percentile(cycle_data['IA'], 25) # higher than 25th percentile of the data
    lo_freq_duration = fs/5  # restrict the analysis to 5-12 Hz
    hi_freq_duration = fs/12

    conditions = ['is_good==1',
                        f'duration_samples<{lo_freq_duration}',
                        f'duration_samples>{hi_freq_duration}',
                        f'max_amp>{amp_thresh}']
    print(len(cycle_data['theta_imf']))
    all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
    
    # Check if any cycles satisfy the conditions
    if all_cycles is None or all_cycles.chain_vect.size == 0:
        print("No cycles satisfy the conditions.")
        return pd.DataFrame(), pd.DataFrame(), []
    
    subset_cycles_df = all_cycles.get_metric_dataframe(subset=True)
    subset_indices = subset_cycles_df['index'].values

    all_cycles_inds = get_cycle_inds(all_cycles, subset_indices)
    cycles_inds = arrange_cycle_inds(all_cycles_inds)

    freqs = imf_frequencies[idx]
    sub_theta, theta, supra_theta = tg_split(freqs, theta_range)
    supra_theta_sig = np.sum(imf.T[supra_theta], axis=0)

    # # Corrected Wavelet Transform Computation
    raw_data=sails.wavelet.morlet(supra_theta_sig, freqs=frequencies, sample_rate=fs, ncycles=5,ret_mode='power', normalise='simple')
    raw_wavelets.append(raw_data)
    supraPlot = scipy.stats.zscore(raw_data, axis=1)
    FPP = bin_tf_to_fpp(cycles_inds, supraPlot, bin_count=19)
    all_FPPs.append(FPP)

    # Compute mode frequency for each cycle
    mode_freqs, entropies = compute_mode_frequency_and_entropy(FPP, frequencies, angles)

    all_waveforms, _ = emd.cycles.phase_align(cycle_data['IP'], cycle_data['theta_imf'],
                                                            cycles=all_cycles.iterate(through='subset'), npoints=100)
    all_waveforms = pd.DataFrame(all_waveforms.T)

    waveforms = pd.concat([waveforms, all_waveforms])

    trial = all_cycles.get_metric_dataframe(subset=True)
    trial['mode_freqs'] = mode_freqs
    trial['entropy'] = entropies
    all_trials = pd.concat([all_trials, trial])

  return waveforms, all_trials, all_FPPs

In [ ]:
fs = 1000
theta_range = [5, 12]

# Field-Field PPC — Per-Interval Approach
# Following Zhang et al. (2019)

**Key difference from v2:** Instead of pooling all theta cycles across all phasic (or tonic) intervals within a trial into one PPC, here we:
1. Compute PPC **per interval** (each phasic/tonic REM episode separately)
2. Average the per-interval PPC matrices within each trial
3. Then average across trials

This gives each interval equal weight, regardless of how many theta cycles it contains.

In [ ]:
def compute_ppc_single_interval(hpc_imf, pfc_lfp, fs=1000,
                                 frequencies=np.arange(15, 141, 1),
                                 n_phase_bins=20, n_cycles_wavelet=5):
    """
    Compute PPC(f, phi) for a single phasic or tonic interval.

    Parameters
    ----------
    hpc_imf : ndarray, shape (n_samples, n_imfs)
        IMFs for one interval.
    pfc_lfp : ndarray, shape (n_samples,)
        PFC LFP for the same interval.

    Returns
    -------
    ppc : ndarray (n_freqs, n_phase_bins) or None
    n_cycles : int
    """
    hpc = np.sum(hpc_imf, axis=1)
    min_len = min(len(hpc), len(pfc_lfp))
    hpc = hpc[:min_len]
    pfc = pfc_lfp[:min_len]
    theta_imf = hpc_imf[:min_len, 5]

    # Detect theta cycles
    cycle_data = get_cycle_data(theta_imf, fs)
    amp_thresh = np.percentile(cycle_data['IA'], 25)
    conditions = [
        'is_good==1',
        f'duration_samples<{fs / 5}',
        f'duration_samples>{fs / 12}',
        f'max_amp>{amp_thresh}'
    ]
    all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
    if all_cycles is None or all_cycles.chain_vect.size == 0:
        return None, 0

    subset_df = all_cycles.get_metric_dataframe(subset=True)
    cycle_inds = arrange_cycle_inds(
        get_cycle_inds(all_cycles, subset_df['index'].values)
    )

    # Complex wavelet transforms
    hpc_cwt = sails.wavelet.morlet(
        hpc, freqs=frequencies, sample_rate=fs,
        ncycles=n_cycles_wavelet, ret_mode='complex', normalise='simple'
    )
    pfc_cwt = sails.wavelet.morlet(
        pfc, freqs=frequencies, sample_rate=fs,
        ncycles=n_cycles_wavelet, ret_mode='complex', normalise='simple'
    )
    cross = hpc_cwt * np.conj(pfc_cwt)

    all_W = []
    for c in range(len(cycle_inds)):
        start, end = cycle_inds[c]
        cycle_len = end - start
        if end > min_len or cycle_len < n_phase_bins:
            continue

        cs_cycle = cross[:, start:end]
        bin_edges = np.linspace(0, cycle_len, n_phase_bins + 1, dtype=int)
        B_k = np.zeros((len(frequencies), n_phase_bins), dtype=complex)
        for pb in range(n_phase_bins):
            b_s, b_e = bin_edges[pb], bin_edges[pb + 1]
            if b_e > b_s:
                B_k[:, pb] = np.mean(cs_cycle[:, b_s:b_e], axis=1)
        all_W.append(np.angle(B_k))

    N = len(all_W)
    if N < 2:
        return None, N

    sum_vec = np.zeros((len(frequencies), n_phase_bins), dtype=complex)
    for W_k in all_W:
        sum_vec += np.exp(1j * W_k)
    ppc = (np.abs(sum_vec)**2 - N) / (N * (N - 1))

    return ppc, N


def compute_ppc_per_interval(hpc_imfs, pfc_lfp_segments, fs=1000,
                              frequencies=np.arange(15, 141, 1),
                              n_phase_bins=20, n_cycles_wavelet=5,
                              min_cycles=2):
    """
    Compute PPC per interval, then average across intervals.

    Parameters
    ----------
    hpc_imfs : list of ndarray
        One IMF array per interval.
    pfc_lfp_segments : list of ndarray
        One PFC LFP per interval.
    min_cycles : int
        Minimum cycles for an interval's PPC to be included.

    Returns
    -------
    avg_ppc : ndarray (n_freqs, n_phase_bins) or None
    sem_ppc : ndarray or None
    n_intervals : int (number of valid intervals used)
    total_cycles : int
    per_interval_ppcs : list of ndarray (individual interval PPCs)
    """
    interval_ppcs = []
    total_cycles = 0

    for idx in range(len(hpc_imfs)):
        ppc, n_cyc = compute_ppc_single_interval(
            hpc_imfs[idx], pfc_lfp_segments[idx], fs=fs,
            frequencies=frequencies, n_phase_bins=n_phase_bins,
            n_cycles_wavelet=n_cycles_wavelet
        )
        if ppc is not None and n_cyc >= min_cycles:
            interval_ppcs.append(ppc)
            total_cycles += n_cyc

    n_intervals = len(interval_ppcs)

    if n_intervals == 0:
        return None, None, 0, total_cycles, []

    arr = np.array(interval_ppcs)
    avg_ppc = np.mean(arr, axis=0)
    sem_ppc = np.std(arr, axis=0) / np.sqrt(n_intervals) if n_intervals > 1 else np.zeros_like(avg_ppc)

    return avg_ppc, sem_ppc, n_intervals, total_cycles, interval_ppcs

## Look into trials — Per-Interval PPC

For each post-trial session, we compute PPC **per phasic/tonic interval** (not pooling cycles across intervals), then average the per-interval PPCs within that trial. Only trials with both valid phasic AND tonic PPC are kept.

In [ ]:
def find_file_local(directory, prefix):
    """Auto-discover a .mat file by prefix (e.g. 'HPC_100', 'PFC_100', 'states')."""
    for fname in os.listdir(directory):
        low = fname.lower()
        if low.startswith(prefix.lower()) and low.endswith('.mat'):
            return os.path.join(directory, fname)
        if prefix == 'states' and 'states' in low and low.endswith('.mat'):
            return os.path.join(directory, fname)
    return None


def compute_per_trial_ppc_v3(rat_ids, base_path='/Users/amir/Desktop/for Abdel/RGS/DatabyCondition',
                             fs=1000, frequencies=np.arange(15, 141, 1),
                             n_phase_bins=20, n_cycles_wavelet=5,
                             min_cycles=2, save_path=None):
    """
    Compute PPC(f, phi) per trial using the per-interval approach.
    
    For each trial's phasic (or tonic) intervals:
      1. Compute PPC separately for each interval via compute_ppc_single_interval
      2. Average the per-interval PPCs within that trial via compute_ppc_per_interval
    
    Only keeps trials that have BOTH valid phasic AND tonic PPC.

    Returns
    -------
    trial_results : list of dict
        Each dict has: rat_id, condition, sd, trial_number, folder,
          phasic_ppc (avg over intervals), phasic_sem, phasic_n_intervals, phasic_n_cycles,
          tonic_ppc, tonic_sem, tonic_n_intervals, tonic_n_cycles,
          frequencies, n_phase_bins
    """
    cfg = globals().get("config", None)
    if cfg is None:
        raise RuntimeError("Define `config` (emd SiftConfig) before calling.")

    preferred_conditions = ["HomeCageHC", "RandomCon", "OverlappingOR", "StableCondOD", "HomeCageCG"]
    conditions = [c for c in preferred_conditions if os.path.isdir(os.path.join(base_path, c))]
    folder_re = re.compile(r'^OS_Ephys_RGS14_Rat(\d+)_\d+_SD(\d+)_([\w-]+)_([\d-]+)$')

    trial_results = []
    skipped_single = 0

    for rat_id in rat_ids:
        print(f"\n{'='*60}")
        print(f"  Rat {rat_id} — per-interval PPC (v3)")
        print(f"{'='*60}")

        for condition_folder in conditions:
            condition_path = os.path.join(base_path, condition_folder)
            rat_folders = []
            for f in os.listdir(condition_path):
                if not os.path.isdir(os.path.join(condition_path, f)):
                    continue
                m = folder_re.match(f)
                if m and int(m.group(1)) == rat_id:
                    rat_folders.append((f, m))

            for rat_folder, m in sorted(rat_folders, key=lambda x: x[0]):
                rat_path = os.path.join(condition_path, rat_folder)
                sd_number = m.group(2)
                condition = m.group(3)

                pt_folders = sorted([
                    f for f in os.listdir(rat_path)
                    if os.path.isdir(os.path.join(rat_path, f)) and re.search(r'Post[-_]Trial\d+', f)
                ])

                for pt_folder in pt_folders:
                    trial_path = os.path.join(rat_path, pt_folder)
                    hpc_file = find_file_local(trial_path, 'HPC_100')
                    pfc_file = find_file_local(trial_path, 'PFC_100')
                    state_file = find_file_local(trial_path, 'states')

                    if not hpc_file or not pfc_file or not state_file:
                        missing = [x for x, v in [('HPC', hpc_file), ('PFC', pfc_file), ('states', state_file)] if not v]
                        print(f"  [SKIP] {condition_folder}/{pt_folder} — missing: {', '.join(missing)}")
                        continue

                    trial_match = re.search(r'Post[-_]Trial(\d+)', pt_folder)
                    trial_number = int(trial_match.group(1)) if trial_match else -1

                    try:
                        lfpHPC_r, hypno_r, _ = get_data(hpc_file, state_file)
                        lfpPFC_r, _, _ = get_data(pfc_file, state_file, type='pfc')

                        try:
                            phasic_int, tonic_int, lfp_raw_r = extract_pt_intervals(
                                lfpHPC_r, hypno_r, fs=fs)
                        except ValueError:
                            print(f"  [SKIP] {condition_folder}/{pt_folder} — no REM")
                            continue

                        has_phasic = phasic_int is not None and len(phasic_int) > 0
                        has_tonic = tonic_int is not None and len(tonic_int) > 0

                        if not has_phasic or not has_tonic:
                            skipped_single += 1
                            print(f"  [SKIP] {condition_folder}/{pt_folder} — "
                                  f"only {'tonic' if has_tonic else 'phasic'} (need both)")
                            continue

                        trial_entry = {
                            'rat_id': rat_id,
                            'condition': condition_folder,
                            'sd': sd_number,
                            'trial_number': trial_number,
                            'folder': pt_folder,
                            'frequencies': frequencies,
                            'n_phase_bins': n_phase_bins,
                            'phasic_ppc': None, 'phasic_sem': None,
                            'phasic_n_intervals': 0, 'phasic_n_cycles': 0,
                            'tonic_ppc': None, 'tonic_sem': None,
                            'tonic_n_intervals': 0, 'tonic_n_cycles': 0,
                        }

                        # --- Phasic (per-interval) ---
                        try:
                            ph_imfs, _, _ = extract_imfs_by_pt_intervals(
                                lfp_raw_r, fs, phasic_int, cfg, return_imfs_freqs=True)
                            pfc_ph = extract_lfp_by_pt_intervals(lfpPFC_r, fs, phasic_int)
                            avg_ppc, sem_ppc, n_int, n_cyc, _ = compute_ppc_per_interval(
                                ph_imfs, pfc_ph, fs=fs,
                                frequencies=frequencies,
                                n_phase_bins=n_phase_bins,
                                n_cycles_wavelet=n_cycles_wavelet,
                                min_cycles=min_cycles)
                            trial_entry['phasic_ppc'] = avg_ppc
                            trial_entry['phasic_sem'] = sem_ppc
                            trial_entry['phasic_n_intervals'] = n_int
                            trial_entry['phasic_n_cycles'] = n_cyc
                        except Exception as e:
                            print(f"  [WARN] Phasic {condition_folder}/{pt_folder}: {e}")

                        # --- Tonic (per-interval) ---
                        try:
                            to_imfs, _, _ = extract_imfs_by_pt_intervals(
                                lfp_raw_r, fs, tonic_int, cfg, return_imfs_freqs=True)
                            pfc_to = extract_lfp_by_pt_intervals(lfpPFC_r, fs, tonic_int)
                            avg_ppc, sem_ppc, n_int, n_cyc, _ = compute_ppc_per_interval(
                                to_imfs, pfc_to, fs=fs,
                                frequencies=frequencies,
                                n_phase_bins=n_phase_bins,
                                n_cycles_wavelet=n_cycles_wavelet,
                                min_cycles=min_cycles)
                            trial_entry['tonic_ppc'] = avg_ppc
                            trial_entry['tonic_sem'] = sem_ppc
                            trial_entry['tonic_n_intervals'] = n_int
                            trial_entry['tonic_n_cycles'] = n_cyc
                        except Exception as e:
                            print(f"  [WARN] Tonic {condition_folder}/{pt_folder}: {e}")

                        # Only keep if BOTH phasic and tonic PPC are valid
                        both_valid = (trial_entry['phasic_ppc'] is not None
                                      and trial_entry['tonic_ppc'] is not None
                                      and trial_entry['phasic_n_intervals'] >= 1
                                      and trial_entry['tonic_n_intervals'] >= 1)

                        if both_valid:
                            trial_results.append(trial_entry)
                            print(f"  [OK]   {condition_folder}/{pt_folder} | "
                                  f"phasic: {trial_entry['phasic_n_intervals']} intervals "
                                  f"({trial_entry['phasic_n_cycles']} cycles), "
                                  f"tonic: {trial_entry['tonic_n_intervals']} intervals "
                                  f"({trial_entry['tonic_n_cycles']} cycles)")
                        else:
                            skipped_single += 1
                            print(f"  [SKIP] {condition_folder}/{pt_folder} — "
                                  f"phasic: {trial_entry['phasic_n_intervals']} int, "
                                  f"tonic: {trial_entry['tonic_n_intervals']} int "
                                  f"(need both >= 1 valid interval)")

                    except Exception as e:
                        print(f"  [ERROR] {condition_folder}/{pt_folder}: {e}")

    print(f"\nTotal trials kept (both phasic & tonic): {len(trial_results)}")
    print(f"Trials skipped (missing one state): {skipped_single}")

    if save_path:
        with open(save_path, 'wb') as f:
            pickle.dump(trial_results, f)
        print(f"Saved to {save_path}")

    return trial_results

### Per trial — multi rat

In [ ]:
# ---- Run per-trial PPC (per-interval approach) ----
selected_rats_trials = [1, 3, 6, 9]  # <-- CHANGE THIS

trial_results = compute_per_trial_ppc_v3(
    rat_ids=selected_rats_trials,
    save_path='ppc_per_trial_results_v3.pkl'
)

In [ ]:
# Quick summary table of all trials
summary_rows = []
for t in trial_results:
    summary_rows.append({
        'Rat': t['rat_id'],
        'Condition': t['condition'],
        'SD': t['sd'],
        'Trial': t['trial_number'],
        'Folder': t['folder'],
        'Phasic intervals': t['phasic_n_intervals'],
        'Phasic cycles': t['phasic_n_cycles'],
        'Tonic intervals': t['tonic_n_intervals'],
        'Tonic cycles': t['tonic_n_cycles'],
    })
trial_summary = pd.DataFrame(summary_rows)
trial_summary

In [ ]:
def plot_trial_ppc_heatmaps(trial_results, rat_id, condition=None, vmin=-0.05, vmax=0.15,
                           save_dir='ppc_trial_figures_v3'):
    """
    Plot PPC(f, phi) heatmaps for individual trials (per-interval averaged).
    Each trial saved as a separate PNG.
    """
    os.makedirs(save_dir, exist_ok=True)

    phase_bins = np.linspace(-180, 180, 21)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2

    trials = [t for t in trial_results if t['rat_id'] == rat_id]
    if condition is not None:
        trials = [t for t in trials if t['condition'] == condition]

    if not trials:
        print(f"No valid trials for Rat {rat_id}" + (f" condition={condition}" if condition else ""))
        return

    saved_files = []
    for t in trials:
        freqs = t['frequencies']
        label = f"{t['condition']}_SD{t['sd']}_Trial{t['trial_number']}"

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'Rat {rat_id} — {t["condition"]}/SD{t["sd"]}/Trial{t["trial_number"]}\n'
                     f'(phasic: {t["phasic_n_intervals"]} int/{t["phasic_n_cycles"]} cyc, '
                     f'tonic: {t["tonic_n_intervals"]} int/{t["tonic_n_cycles"]} cyc)',
                     fontsize=12, fontweight='bold')

        # Phasic
        im = axes[0].pcolormesh(phase_centers, freqs, t['phasic_ppc'],
                                cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=axes[0], label='PPC')
        axes[0].set_title(f'Phasic ({t["phasic_n_intervals"]} intervals)')
        axes[0].set_ylabel('Frequency (Hz)')
        axes[0].set_xlabel('Theta phase (deg)')

        # Tonic
        im = axes[1].pcolormesh(phase_centers, freqs, t['tonic_ppc'],
                                cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=axes[1], label='PPC')
        axes[1].set_title(f'Tonic ({t["tonic_n_intervals"]} intervals)')
        axes[1].set_ylabel('Frequency (Hz)')
        axes[1].set_xlabel('Theta phase (deg)')

        # Difference
        diff = t['phasic_ppc'] - t['tonic_ppc']
        vd = max(np.max(np.abs(diff)) * 0.8, 0.01)
        im = axes[2].pcolormesh(phase_centers, freqs, diff,
                                cmap='RdBu_r', shading='auto', vmin=-vd, vmax=vd)
        plt.colorbar(im, ax=axes[2], label='$\\Delta$PPC')
        axes[2].set_title('Phasic - Tonic')
        axes[2].set_ylabel('Frequency (Hz)')
        axes[2].set_xlabel('Theta phase (deg)')

        plt.tight_layout()
        fname = os.path.join(save_dir, f'Rat{rat_id}_{label}_heatmap.png')
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        saved_files.append(fname)
        plt.close(fig)

    print(f"Saved {len(saved_files)} heatmap figures to: {save_dir}/")
    for f in saved_files:
        print(f"  {f}")


def plot_trial_ppc_curves(trial_results, rat_id, condition=None, freq_range=(15, 140),
                          save_dir='ppc_trial_figures_v3'):
    """
    Plot phase-averaged PPC(f) for each trial as a separate PNG.
    Includes SEM shading from per-interval variability.
    """
    os.makedirs(save_dir, exist_ok=True)

    trials = [t for t in trial_results if t['rat_id'] == rat_id]
    if condition is not None:
        trials = [t for t in trials if t['condition'] == condition]

    saved_files = []
    for t in trials:
        freqs = t['frequencies']
        mask = (freqs >= freq_range[0]) & (freqs <= freq_range[1])
        label = f"{t['condition']}_SD{t['sd']}_Trial{t['trial_number']}"

        fig, ax = plt.subplots(figsize=(10, 4))

        # Phasic with SEM
        ppc_ph = np.mean(t['phasic_ppc'], axis=1)
        ax.plot(freqs[mask], ppc_ph[mask], color='red', linewidth=1.5,
                label=f'Phasic ({t["phasic_n_intervals"]} int, {t["phasic_n_cycles"]} cyc)')
        if t['phasic_sem'] is not None and t['phasic_n_intervals'] > 1:
            sem_ph = np.mean(t['phasic_sem'], axis=1)
            ax.fill_between(freqs[mask], ppc_ph[mask] - sem_ph[mask], ppc_ph[mask] + sem_ph[mask],
                            color='red', alpha=0.2)

        # Tonic with SEM
        ppc_to = np.mean(t['tonic_ppc'], axis=1)
        ax.plot(freqs[mask], ppc_to[mask], color='blue', linewidth=1.5,
                label=f'Tonic ({t["tonic_n_intervals"]} int, {t["tonic_n_cycles"]} cyc)')
        if t['tonic_sem'] is not None and t['tonic_n_intervals'] > 1:
            sem_to = np.mean(t['tonic_sem'], axis=1)
            ax.fill_between(freqs[mask], ppc_to[mask] - sem_to[mask], ppc_to[mask] + sem_to[mask],
                            color='blue', alpha=0.2)

        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('Frequency (Hz)')
        ax.set_ylabel('PPC (phase-averaged)')
        ax.set_title(f'Rat {rat_id} — {t["condition"]}/SD{t["sd"]}/Trial{t["trial_number"]}')
        ax.legend()
        sns.despine()
        plt.tight_layout()

        fname = os.path.join(save_dir, f'Rat{rat_id}_{label}_curve.png')
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        saved_files.append(fname)
        plt.close(fig)

    print(f"Saved {len(saved_files)} curve figures to: {save_dir}/")
    for f in saved_files:
        print(f"  {f}")

In [ ]:
# ---- Browse a specific rat's trials ----
# Heatmaps: each trial saved as separate PNG
plot_trial_ppc_heatmaps(trial_results, rat_id=9)  # condition='HomeCageHC' to filter

In [ ]:
# ---- PPC(f) curves for one rat's trials ----
plot_trial_ppc_curves(trial_results, rat_id=9)  # condition='HomeCageHC' to filter

### Trial-averaged PPC (per-interval approach)
Average PPC(f, phi) across trials. Each trial's PPC is itself an average over its phasic/tonic intervals, so intervals are weighted equally within each trial, and trials are weighted equally in the grand average.

In [ ]:
def average_trial_ppc(trial_results, rat_id=None, condition=None, min_intervals=1):
    """
    Average PPC(f, phi) across trials. All trials already have both phasic & tonic.

    Parameters
    ----------
    trial_results : list of dict
        Output of compute_per_trial_ppc_v3.
    rat_id : int or None
        Filter by rat. None = all rats.
    condition : str or None
        Filter by condition folder name.
    min_intervals : int
        Minimum number of valid intervals required for BOTH states.

    Returns
    -------
    result : dict with keys: frequencies, n_phase_bins, n_trials,
        phasic_avg, phasic_sem, phasic_n_trials, phasic_per_rat,
        tonic_avg, tonic_sem, tonic_n_trials, tonic_per_rat
    """
    trials = trial_results
    if rat_id is not None:
        trials = [t for t in trials if t['rat_id'] == rat_id]
    if condition is not None:
        trials = [t for t in trials if t['condition'] == condition]

    trials = [t for t in trials
              if t['phasic_n_intervals'] >= min_intervals
              and t['tonic_n_intervals'] >= min_intervals]

    rat_ids_present = sorted(set(t['rat_id'] for t in trials)) if trials else []

    result = {
        'frequencies': trials[0]['frequencies'] if trials else None,
        'n_phase_bins': trials[0]['n_phase_bins'] if trials else None,
        'n_trials': len(trials),
    }

    for label in ['phasic', 'tonic']:
        all_ppcs = [t[f'{label}_ppc'] for t in trials]

        if all_ppcs:
            arr = np.array(all_ppcs)
            result[f'{label}_avg'] = np.mean(arr, axis=0)
            result[f'{label}_sem'] = np.std(arr, axis=0) / np.sqrt(len(arr))
            result[f'{label}_n_trials'] = len(arr)
        else:
            result[f'{label}_avg'] = None
            result[f'{label}_sem'] = None
            result[f'{label}_n_trials'] = 0

        per_rat = {}
        for rid in rat_ids_present:
            rat_ppcs = [t[f'{label}_ppc'] for t in trials if t['rat_id'] == rid]
            per_rat[rid] = np.mean(rat_ppcs, axis=0) if rat_ppcs else None
        result[f'{label}_per_rat'] = per_rat

    return result


def plot_trial_averaged_ppc_heatmap(avg_result, title='Trial-Averaged PPC',
                                     vmin=-0.05, vmax=0.15, save_path=None):
    """Plot trial-averaged PPC(f, phi) heatmaps: phasic, tonic, difference."""
    phase_bins = np.linspace(-180, 180, 21)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2
    freqs = avg_result['frequencies']
    n_tr = avg_result['n_trials']

    ppc_ph = avg_result['phasic_avg']
    ppc_to = avg_result['tonic_avg']

    if ppc_ph is None or ppc_to is None:
        print("Not enough trials with both phasic & tonic to plot.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{title} ({n_tr} trials)', fontsize=14, fontweight='bold')

    im = axes[0].pcolormesh(phase_centers, freqs, ppc_ph,
                             cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=axes[0], label='PPC')
    axes[0].set_title(f'Phasic ({avg_result["phasic_n_trials"]} trials)')
    axes[0].set_ylabel('Frequency (Hz)')
    axes[0].set_xlabel('Theta phase (deg)')

    im = axes[1].pcolormesh(phase_centers, freqs, ppc_to,
                             cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=axes[1], label='PPC')
    axes[1].set_title(f'Tonic ({avg_result["tonic_n_trials"]} trials)')
    axes[1].set_ylabel('Frequency (Hz)')
    axes[1].set_xlabel('Theta phase (deg)')

    diff = ppc_ph - ppc_to
    vd = max(np.max(np.abs(diff)) * 0.8, 0.01)
    im = axes[2].pcolormesh(phase_centers, freqs, diff,
                             cmap='RdBu_r', shading='auto', vmin=-vd, vmax=vd)
    plt.colorbar(im, ax=axes[2], label='$\\Delta$PPC')
    axes[2].set_title('Phasic - Tonic')
    axes[2].set_ylabel('Frequency (Hz)')
    axes[2].set_xlabel('Theta phase (deg)')

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    plt.show()


def plot_trial_averaged_ppc_curves(avg_result, title='Trial-Averaged PPC(f)',
                                    freq_range=(15, 140), save_path=None):
    """Plot trial-averaged phase-averaged PPC(f) with SEM shading."""
    freqs = avg_result['frequencies']
    if freqs is None:
        print("No data to plot.")
        return
    mask = (freqs >= freq_range[0]) & (freqs <= freq_range[1])

    fig, ax = plt.subplots(figsize=(10, 5))

    for label, color in [('phasic', 'red'), ('tonic', 'blue')]:
        avg = avg_result[f'{label}_avg']
        sem = avg_result[f'{label}_sem']
        n = avg_result[f'{label}_n_trials']
        if avg is not None:
            ppc_f = np.mean(avg, axis=1)
            sem_f = np.mean(sem, axis=1)
            ax.plot(freqs[mask], ppc_f[mask], color=color, linewidth=2,
                    label=f'{label.capitalize()} ({n} trials)')
            ax.fill_between(freqs[mask], ppc_f[mask] - sem_f[mask], ppc_f[mask] + sem_f[mask],
                            color=color, alpha=0.2)

    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PPC (phase-averaged)')
    ax.set_title(title)
    ax.legend()
    sns.despine()
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    plt.show()


def plot_trial_averaged_per_rat(trial_results, rat_ids=None, min_intervals=1,
                                 vmin=-0.05, vmax=0.15, save_dir=None):
    """
    Plot trial-averaged PPC for each rat separately, then grand average.
    """
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    if rat_ids is None:
        rat_ids = sorted(set(t['rat_id'] for t in trial_results))

    # Per-rat plots
    for rid in rat_ids:
        avg = average_trial_ppc(trial_results, rat_id=rid, min_intervals=min_intervals)
        if avg['n_trials'] == 0:
            print(f"Rat {rid}: no trials with both states >= {min_intervals} intervals")
            continue

        sp_hm = os.path.join(save_dir, f'Rat{rid}_trial_avg_heatmap.png') if save_dir else None
        sp_cv = os.path.join(save_dir, f'Rat{rid}_trial_avg_curve.png') if save_dir else None

        plot_trial_averaged_ppc_heatmap(
            avg, title=f'Rat {rid} — Trial-Averaged PPC (per-interval)',
            vmin=vmin, vmax=vmax, save_path=sp_hm)
        plot_trial_averaged_ppc_curves(
            avg, title=f'Rat {rid} — Trial-Averaged PPC(f) (per-interval)',
            save_path=sp_cv)

    # Grand average across all selected rats
    if len(rat_ids) > 1:
        trials_filtered = [t for t in trial_results if t['rat_id'] in rat_ids]
        avg_all = average_trial_ppc(trials_filtered, rat_id=None, min_intervals=min_intervals)

        if avg_all['n_trials'] == 0:
            print(f"No trials across rats with both states >= {min_intervals} intervals")
            return

        sp_hm = os.path.join(save_dir, 'GrandAvg_trial_avg_heatmap.png') if save_dir else None
        sp_cv = os.path.join(save_dir, 'GrandAvg_trial_avg_curve.png') if save_dir else None

        plot_trial_averaged_ppc_heatmap(
            avg_all,
            title=f'Grand Average — Per-Interval PPC (n={len(rat_ids)} rats)',
            vmin=vmin, vmax=vmax, save_path=sp_hm)
        plot_trial_averaged_ppc_curves(
            avg_all,
            title=f'Grand Average — Per-Interval PPC(f) (n={len(rat_ids)} rats)',
            save_path=sp_cv)

        # Per-rat overlay figure
        freqs = avg_all['frequencies']
        fmask = (freqs >= 15) & (freqs <= 140)

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        fig.suptitle('Per-Rat Trial-Averaged PPC(f) (per-interval approach)',
                     fontsize=14, fontweight='bold')

        for label, color, ax in [('phasic', 'red', axes[0]), ('tonic', 'blue', axes[1])]:
            per_rat = avg_all[f'{label}_per_rat']
            curves = []
            for rid in sorted(per_rat.keys()):
                if per_rat[rid] is not None:
                    ppc_f = np.mean(per_rat[rid], axis=1)
                    ax.plot(freqs[fmask], ppc_f[fmask], alpha=0.5, linewidth=1,
                            label=f'Rat {rid}')
                    curves.append(ppc_f[fmask])
            if curves:
                arr = np.array(curves)
                mean_c = np.mean(arr, axis=0)
                sem_c = np.std(arr, axis=0) / np.sqrt(len(arr))
                ax.plot(freqs[fmask], mean_c, color='black', linewidth=2.5, label='Mean')
                ax.fill_between(freqs[fmask], mean_c - sem_c, mean_c + sem_c,
                                color='gray', alpha=0.3)
            ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
            ax.set_xlabel('Frequency (Hz)')
            ax.set_ylabel('PPC')
            ax.set_title(f'{label.capitalize()} REM')
            ax.legend(fontsize=7, loc='upper right')
            sns.despine(ax=ax)

        plt.tight_layout()
        if save_dir:
            sp = os.path.join(save_dir, 'PerRat_overlay_curves.png')
            fig.savefig(sp, dpi=150, bbox_inches='tight')
            print(f"Saved to {sp}")
        plt.show()

In [ ]:
# ---- Trial-averaged PPC for a single rat ----
plot_trial_averaged_per_rat(
    trial_results,
    rat_ids=[1],          # <-- single rat
    min_intervals=1,
    save_dir='ppc_trial_avg_figures_v3'
)

In [ ]:
# ---- Trial-averaged PPC across all rats (grand average) ----
plot_trial_averaged_per_rat(
    trial_results,
    rat_ids=[1, 3, 6, 9],   # <-- all rats; shows per-rat + grand avg
    min_intervals=1,
    save_dir='ppc_trial_avg_figures_v3'
)